In [43]:
class A:
    def foo(self):
        return "A.foo called"

def bar(a: A):
    return "bar called"

obj = A()

obj.foo()
y=bar(obj)
y

'bar called'

In [19]:
class Order:
    def total(self):
        return 100

    def due(self):
        return self.total()

x = Order()
x.due()

100

In [ ]:
class Promo:
    def discount(self, order):
        return 10

class Order:
    def __init__(self, promo):
        self.promo = promo

    def due(self):
        return 100 - self.promo.discount(self) # self.promo, gives u a promo object, the other self is a order obj
        '''
        Python passes self.promo as the first argument automatically.
        The extra self inside parentheses is the Order instance.
        '''

x = Order(Promo())
x.due()

90

In [48]:
def promo(order):  # Promo → reads Order
    return 10

class Order:  
    def __init__(self, promo):  # Order → calls Promo
        self.promo = promo

    def due(self):
        return 100 - self.promo(self)  


x = Order(promo)
x.due()

90

In [62]:
class Item:
    def total(self):
        return 5

class Order:
    def __init__(self, items):
        self.items = items

    def total(self):
        return sum(item.total() for item in self.items)


x = Order([Item(),Item()])
x.total()

10

.globals() to check for all variables within a file

In [ ]:
# 1. We write our functions, making sure they all end with '_format'
def uppercase_format(text):
    return text.upper()

def dash_format(text):
    return text.replace(" ", "-")

def normal_function(text):
    return "This won't be grabbed because of its name"

# 2. We use globals() to automatically hunt down the right functions!
# This loops through everything in the file.
all_formatters = [
    func for name, func in globals().items() 
    if name.endswith('_format')
]
all_formatters

# Cons: It's a bit "hacky." If you accidentally name a standard variable my_cool_format = "Hello", the code might grab that string instead of a function and crash.

[<function __main__.uppercase_format(text)>,
 <function __main__.dash_format(text)>]

Instead of relying on naming conventions (_format), we put all of our strategy functions into a completely separate file (module) called formatters.py.

Then, we use Python's built-in inspect library to peek into that file and say: "Give me every single function you find in that room."

In [ ]:
import inspect
import formatters  # This is the separate file containing our functions

# Go into the 'formatters' module and grab anything that is a function
all_formatters = [
    func for name, func in inspect.getmembers(formatters, inspect.isfunction)
]

#Cons: Every function in that file will be grabbed. If you write a small helper function in that file that isn't meant to be a formatter, it gets grabbed anyway and breaks your master function.

Decorators (The "Sign-Up Sheet" Trick)
This is the cleanest and most "Pythonic" way to solve the proble

In [4]:
# 1. The empty sign-up sheet
all_formatters = []

# 2. The Decorator (The Stamp)
def register(func):
    all_formatters.append(func) # Add the function to the list
    return func                 # Return it unchanged so it still works normally

# 3. Apply the stamp to the functions you want to include!
@register
def uppercase(text):
    return text.upper()

@register
def spaces_to_dashes(text):
    return text.replace(" ", "-")

def do_not_include(text):
    return "I don't have the stamp, so I am not in the list!"

all_formatters

[<function __main__.uppercase(text)>,
 <function __main__.spaces_to_dashes(text)>]

Python treats functions as first-class objects. You can pass a function into a list, assign it to a variable, and execute it later, all without ever creating a Class to hold it.

Because of this, creating an entire Command base class with a single execute() method is usually a waste of time in Python. "Commands are an object-oriented replacement for callbacks." If you can just use a callback (a simple function), do that instead!

### The folllowing is an example of command, before and after

In [5]:
# invoke internal functionality: button to do something, it does decoupling but it focus on encapsulation for queueing or ordering or so

from abc import ABC, abstractmethod
import queue


class Command(ABC):
    def __init__(self, command_id):
        self.command_id = command_id
    @abstractmethod
    def excute(self):
        print(f'excuting {self}')
    
class OrderFood(Command):
    def excute(self):
        print(f'ordering food: {self.command_id}')

class PayFood(Command):
    def excute(self):
        print(f'paying for food: {self.command_id}')
    

class ProcessCommand:
    queue = []

    def add_to_queue(self,command: Command):
        self.queue.append(command)

    def process(self):
        [item.excute() for item in self.queue]
        self.queue = []


if __name__ == "__main__":
    processor = ProcessCommand()
    processor.add_to_queue(OrderFood('Botato'))
    processor.add_to_queue(PayFood('Botato'))

    processor.process()



ordering food: Botato
paying for food: Botato


In [6]:
# 1. No more Abstract Base Classes (ABC)!
# We replace the Command classes with simple functions that return "closures". 
# The inner function remembers the 'item_name' it was created with.

def order_food(item_name: str):
    def execute_action():
        print(f'ordering food: {item_name}')
    return execute_action  # Notice we return the function, we don't call it yet!

def pay_food(item_name: str):
    def execute_action():
        print(f'paying for food: {item_name}')
    return execute_action


# 2. The MacroCommand (Your ProcessCommand)
class ProcessCommand:
    def __init__(self):
        # Good practice: Initialize the queue inside __init__, not as a class variable
        self.queue = []

    def add_to_queue(self, command_func):
        self.queue.append(command_func)

    # 3. Using __call__ instead of process()
    # This allows the object itself to be called like a function using ()
    def __call__(self):
        for command in self.queue:
            command()  # Just call the function directly! No .execute() needed.
        self.queue = []


if __name__ == "__main__":
    # 1. Setup the queue
    processor = ProcessCommand()
    
    # 2. Add functions to the queue
    processor.add_to_queue(order_food('Botato'))
    processor.add_to_queue(pay_food('Botato'))

    # 3. Process the queue 
    # Because we used __call__, we treat 'processor' like it is a function!
    processor()

ordering food: Botato
paying for food: Botato


Why is this better?
Less Boilerplate: You completely deleted the Command interface class.

Less typing: You don't have to write def __init__(self, command_id): over and over again for every single command subclass.

Flexibility: Your ProcessCommand queue now accepts any function that takes no arguments. You don't have to force other developers to inherit from your specific Command base class to use your queue.